In the previous notebook models_citywide.ipynb, we found that the Prophet and NeuralProphet models performed well in terms of forecasting rat sightings by day citywide. In this notebook, we will do some more feature engineering and hyperparameter tuning to obtain a more optimal model.

Because we wish this to be reusable, we will write things for Prophet and NeuralProphet separately. 

# Import Packages

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import datetime

from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error
from sklearn.linear_model import LinearRegression

from prophet import Prophet
from pandas.tseries.holiday import USFederalHolidayCalendar
from prophet.diagnostics import cross_validation, performance_metrics
from prophet.plot import plot_cross_validation_metric
from prophet.plot import add_changepoints_to_plot
from prophet.plot import plot_plotly, plot_components_plotly
import itertools

import warnings
from statsmodels.tools.sm_exceptions import ConvergenceWarning
warnings.simplefilter('ignore', ConvergenceWarning)

import optuna
from neuralprophet import NeuralProphet
import logging


# Prophet

## Import the data

In [ ]:
# this is the time series split we will work with
tscv = TimeSeriesSplit(gap=0, max_train_size=None, n_splits=26, test_size=14)


# we import the data and clean it for future use
rs = pd.read_csv('../scr/data/cleaned_rat_sightings_data/all_cleaned_rat_sightings.csv')
rs['created_date'] = pd.to_datetime(rs['created_date']) 
rs['closed_date'] = pd.to_datetime(rs['closed_date'])
rs['resolution_action_updated_date'] = pd.to_datetime(rs['resolution_action_updated_date'])
# mark cutoff dates, and also rename columns
rs = rs[rs['created_date']<'2026-03-01']
rs = rs[rs['created_date']>='2020-01-01']
rs = rs.groupby([rs['created_date'].dt.date]).size().reset_index(name='count')
rs.rename(columns={'created_date': 'ds', 'count': 'y'}, inplace=True)
# This is should be 2251 which equals the number of days from 2020-01-01 to 2026-02-28
print(len(rs)==2251)


## Prepare Prophet

In [ ]:
date_range = pd.date_range(start="2020-01-01", end="2026-02-28")

# Generate US federal holidays
calendar = USFederalHolidayCalendar()
holidays = calendar.holidays(start=date_range.min(), end=date_range.max())

federal_holidays = pd.DataFrame({
    'holiday': 'federal_us',
    'ds': pd.to_datetime(holidays),
    'lower_window': 0,
    'upper_window': 1})

holidays = federal_holidays

In [ ]:
rs_saved = rs.copy()
df = rs.copy()

## Hyperparameter Tuning for Prophet

In [ ]:
import logging
logging.getLogger("cmdstanpy").setLevel(logging.WARNING)

In [ ]:
## To tune for hyperparameters, add more possible parameters to the dictionary below and add more values to it.
## So far, the I've been able to get is {'changepoint_prior_scale': 0.1, 'seasonality_prior_scale': 5}

init_days = '2054 days'
cv_period = '14 days'
forecast_horizon = '14 days'

param_grid = {  
    'changepoint_prior_scale': [0.1],
    'seasonality_prior_scale': [5],
}

# Generate all combinations of parameters
all_params = [dict(zip(param_grid.keys(), v)) for v in itertools.product(*param_grid.values())]
rmses = []  # Store the RMSEs for each params here
performance = []

# Use cross validation to evaluate all parameters
for params in all_params:
    params['holidays'] = holidays
    m = Prophet(**params).fit(df)  # Fit model with given params
    df_cv = cross_validation(m, initial = init_days, period=cv_period, horizon = forecast_horizon)
    df_p = performance_metrics(df_cv, rolling_window=14)
    performance.append(df_p)
    rmses.append(df_p['rmse'].values[0])

# Find the best parameters
tuning_results = pd.DataFrame(all_params)
tuning_results['rmse'] = rmses

best_params = all_params[np.argmin(rmses)]

In [ ]:
new_performance = pd.concat(performance, ignore_index=True)

# Round numeric columns for readability
numeric_cols = ["mse", "rmse", "mae", "mape", "mdape", "smape", "coverage"]
new_performance[numeric_cols] = new_performance[numeric_cols].round(4)


new_performance

## Train the Model

In [ ]:
m = Prophet(**best_params)
m.add_country_holidays(country_name='US')
m.fit(df)
future = m.make_future_dataframe(periods=14)
forecast = m.predict(future)

## Plots and Evaluation of the Model

In [ ]:
fig1 = plot_plotly(m, forecast)
fig1.show()

fig2 = plot_components_plotly(m, forecast)
fig2.show()

# Neural Prophet

## Import Packages

In [3]:
np.NaN = np.nan


# the following packages are meant to turn off a bunch of the warnings and ERRORs that pop up while running NeuralProphet.
# the errors that do show up are not all that important and a lot is due to outdated packages.
import warnings
import logging

warnings.filterwarnings("ignore")

logging.getLogger("neuralprophet").setLevel(logging.ERROR)
logging.getLogger("pytorch_lightning").setLevel(logging.ERROR)
logging.getLogger("NP").setLevel(logging.ERROR)

## Import Data

In [4]:
rs = pd.read_csv('../scr/data/cleaned_rat_sightings_data/all_cleaned_rat_sightings.csv')
rs['created_date'] = pd.to_datetime(rs['created_date']) 
rs['closed_date'] = pd.to_datetime(rs['closed_date'])
rs['resolution_action_updated_date'] = pd.to_datetime(rs['resolution_action_updated_date'])
# mark cutoff dates, and also rename columns
rs = rs[rs['created_date']<'2026-03-01']
rs = rs[rs['created_date']>='2020-01-01']
rs = rs.groupby([rs['created_date'].dt.date]).size().reset_index(name='count')
rs.rename(columns={'created_date': 'ds', 'count': 'y'}, inplace=True)


In [5]:
## Add weather data.
import requests

lat, lon = 40.7831, -73.9712
start = "2020-01-01"
end   = "2026-02-28"

url = (
    "https://archive-api.open-meteo.com/v1/archive"
    f"?latitude={lat}&longitude={lon}"
    f"&start_date={start}&end_date={end}"
    "&daily=temperature_2m_max,temperature_2m_min,temperature_2m_mean,"
    "apparent_temperature_max,apparent_temperature_min,apparent_temperature_mean,"
    "precipitation_sum,snowfall_sum"
    "&timezone=America/New_York"
)

response = requests.get(url)
data = response.json()

if 'error' in data:
    nd = pd.read_csv("weatherdata.csv")
    nd = nd.set_index('date')
    wd = nd
    
else:
    wd = pd.DataFrame(data["daily"])
    wd["date"] = pd.to_datetime(wd["time"])
    wd = wd.set_index("date")

## Optuna for Hyperparameter Tuning for Neural Prophet

In [6]:
# Suppress cmdstanpy info logs
logging.getLogger("cmdstanpy").setLevel(logging.WARNING)


regressed_features = ['apparent_temperature_max', 'apparent_temperature_min', 'snowfall_sum']


wd = wd.reset_index(drop=True).rename(columns={"time": "ds"})
wd["ds"] = pd.to_datetime(wd["ds"])
rs["ds"] = pd.to_datetime(rs["ds"])

rs = rs.merge(
    wd[['ds'] + regressed_features],
    on="ds",
    how="left"
)

tscv = TimeSeriesSplit(gap=0, max_train_size=None, n_splits=2, test_size=14)


def objective(trial):
    regressor_lags = {
        'apparent_temperature_max': trial.suggest_int('lag_temp_max', 1, 30),
        'apparent_temperature_min': trial.suggest_int('lag_temp_min', 1, 30),
        'snowfall_sum': trial.suggest_int('lag_snowfall', 1, 7),
    }
    n_lags = trial.suggest_int('n_lags', 1, 21)
    epochs = trial.suggest_int('epochs', 50, 200)
    learning_rate = trial.suggest_float('learning_rate', 0.001, 0.5, log=True)
    batch_size = trial.suggest_int('batch_size', 20, 128)
    ar_reg = trial.suggest_float('ar_reg', 0.5, 3)
    fold_rmses = []
    for i, (train_idx, test_idx) in enumerate(tscv.split(rs)):

        train = rs.iloc[train_idx].copy()
        test = rs.iloc[test_idx].copy()
        
        existing_regressors = [col for col in regressed_features if col in train.columns]
        train = train.dropna(subset=["y"] + existing_regressors)
        test = test.dropna(subset=existing_regressors)
        
        # Skip fold if too few rows
        if len(train) < 20 or len(test) < 1:
            continue
        
        model = NeuralProphet(
            yearly_seasonality=True,
            weekly_seasonality=True,
            n_lags=n_lags,
            epochs=epochs,
            ar_reg = ar_reg,
            accelerator="auto",   # uses GPU if available
            learning_rate=learning_rate,
            batch_size=batch_size
        )
        model.add_country_holidays(country_name="US")
        for col in existing_regressors:
            model.add_lagged_regressor(col, n_lags=regressor_lags[col])
        
        model.fit(train, freq="D", progress="off")
        future = pd.concat([
            train[['ds','y'] + existing_regressors],
            test[['ds','y']].merge(wd[['ds'] + existing_regressors], on="ds", how="left")
        ])
        future = future.dropna(subset=existing_regressors)
        forecast = model.predict(future)
        
        y_pred = forecast["yhat1"].iloc[-len(test):].values
        y_true = test["y"].values
        
        rmse = np.sqrt(mean_squared_error(y_true, y_pred))
        fold_rmses.append(rmse)

        intermediate_score = np.mean(fold_rmses)
        trial.report(intermediate_score, i)
        if trial.should_prune():
            raise optuna.TrialPruned()
        
    return np.mean(fold_rmses) if fold_rmses else float("inf")

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=10)  # adjust n_trials as needed



best_params = study.best_params

[I 2026-03-09 00:49:38,912] A new study created in memory with name: no-name-8cec7e22-2bc1-4689-b83d-f6a6258a5d13


Training: 0it [00:00, ?it/s]

Predicting: 26it [00:00, ?it/s]

Training: 0it [00:00, ?it/s]

Predicting: 26it [00:00, ?it/s]

...

Predicting: 62it [00:00, ?it/s]

[I 2026-03-09 00:56:59,139] Trial 8 pruned. 


Training: 0it [00:00, ?it/s]

Predicting: 18it [00:00, ?it/s]

[I 2026-03-09 00:57:24,717] Trial 9 pruned. 


In [7]:
print("Best Parameters", best_params)
print("Best RMSE:", study.best_value)

Best Parameters {'lag_temp_max': 25, 'lag_temp_min': 11, 'lag_snowfall': 2, 'n_lags': 13, 'epochs': 102, 'learning_rate': 0.19572567685774558, 'batch_size': 86, 'ar_reg': 2.9258881396448992}
Best RMSE: 10.15682407695249


We write this down since it took a lot of effort (a 3 hours+ run).

Best Parameters {'lag_temp_max': 30, 'lag_temp_min': 5, 'lag_snowfall': 1, 'n_lags': 16, 'epochs': 60, 'learning_rate': 0.2986532324899507, 'batch_size': 128, 'ar_reg': 2.132925719127823}
Best RMSE: 7.787986585116568

In [8]:
# best_params =  {'lag_temp_max': 30, 'lag_temp_min': 5, 'lag_snowfall': 1, 'n_lags': 16, 'epochs': 60, 'learning_rate': 0.2986532324899507, 'batch_size': 128, 'ar_reg': 2.132925719127823}

## Check for Improvement of Model

In [9]:
rs = pd.read_csv('../scr/data/cleaned_rat_sightings_data/all_cleaned_rat_sightings.csv')
rs['created_date'] = pd.to_datetime(rs['created_date']) 
rs['closed_date'] = pd.to_datetime(rs['closed_date'])
rs['resolution_action_updated_date'] = pd.to_datetime(rs['resolution_action_updated_date'])
# mark cutoff dates, and also rename columns
rs = rs[rs['created_date']<'2026-03-01']
rs = rs[rs['created_date']>='2020-01-01']
rs = rs.groupby([rs['created_date'].dt.date]).size().reset_index(name='count')
rs.rename(columns={'created_date': 'ds', 'count': 'y'}, inplace=True)

## Add weather data.
import requests

lat, lon = 40.7831, -73.9712
start = "2020-01-01"
end   = "2026-02-28"

url = (
    "https://archive-api.open-meteo.com/v1/archive"
    f"?latitude={lat}&longitude={lon}"
    f"&start_date={start}&end_date={end}"
    "&daily=temperature_2m_max,temperature_2m_min,temperature_2m_mean,"
    "apparent_temperature_max,apparent_temperature_min,apparent_temperature_mean,"
    "precipitation_sum,snowfall_sum"
    "&timezone=America/New_York"
)

response = requests.get(url)
data = response.json()


if 'error' in data:
    nd = pd.read_csv("weatherdata.csv")
    nd = nd.set_index('date')
    wd = nd
    
else:
    wd = pd.DataFrame(data["daily"])
    wd["date"] = pd.to_datetime(wd["time"])
    wd = wd.set_index("date")

In [10]:
tscv = TimeSeriesSplit(gap=0, max_train_size=None, n_splits=26, test_size=14)

regressed_features = ['apparent_temperature_max', 'apparent_temperature_min','snowfall_sum']


wd = wd.reset_index(drop=True).rename(columns={"time": "ds"})
wd["ds"] = pd.to_datetime(wd["ds"])
rs["ds"] = pd.to_datetime(rs["ds"])

rs = rs.merge(
    wd[['ds'] + regressed_features],
    on="ds",
    how="left"
)

lags_for_regressed_features = dict()
lags_for_regressed_features['apparent_temperature_max'] = best_params['lag_temp_max']
lags_for_regressed_features['apparent_temperature_min'] = best_params['lag_temp_min']
lags_for_regressed_features['snowfall_sum'] = best_params['lag_snowfall']


results = []

for i, (train_index, test_index) in enumerate(tscv.split(rs)):

    train = rs.iloc[train_index].copy()
    train = train.dropna(subset=["y"])

    test = rs.iloc[test_index].copy()


    model = NeuralProphet(yearly_seasonality=True, 
                          weekly_seasonality=True, 
                          learning_rate = best_params['learning_rate'],
                          epochs = best_params['epochs'],
                          n_lags= best_params['n_lags'],
                          ar_reg=best_params['ar_reg'],
                          accelerator="auto",   # uses GPU if available
                          batch_size= best_params['batch_size']
                          )
    model = model.add_country_holidays(country_name="US")
    for column in regressed_features:
        model.add_lagged_regressor(column, n_lags=lags_for_regressed_features[column])
        
    model.fit(train, freq="D", progress="off")

    # build dataframe containing future regressors
    future = pd.concat([train[['ds','y'] + regressed_features], test[['ds','y']].merge(wd[['ds'] + regressed_features], on="ds", how="left")])
    forecast = model.predict(future)

    y_pred = forecast["yhat1"].iloc[-len(test):].values
    y_true = test["y"].values

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = mean_absolute_percentage_error(y_true, y_pred)

    results.append({"fold": i, "rmse": rmse, "mape": mape})

neural_prophet_results_df = pd.DataFrame(results)
neural_prophet_results_df.loc["mean"] = ["mean", neural_prophet_results_df["rmse"].mean(), neural_prophet_results_df["mape"].mean()]
neural_prophet_results_df

Training: 0it [00:00, ?it/s]

Predicting: 22it [00:00, ?it/s]

Training: 0it [00:00, ?it/s]

Predicting: 22it [00:00, ?it/s]

Training: 0it [00:00, ?it/s]

...

Training: 0it [00:00, ?it/s]

Predicting: 26it [00:00, ?it/s]

Training: 0it [00:00, ?it/s]

Predicting: 26it [00:00, ?it/s]

,fold,rmse,mape
0,0,12.680303,0.226392
1,1,6.812259,0.083738
2,2,8.506512,0.133636
3,3,10.370989,0.156853
4,4,11.441512,0.153468
5,5,11.704109,0.155755
6,6,12.522116,0.155007
7,7,12.467644,0.153877
8,8,11.720065,0.153808
9,9,9.252438,0.109665


## Train the Model

In [11]:
model = NeuralProphet(yearly_seasonality=True, 
                          weekly_seasonality=True, 
                          # batch_size = best_params['batch_size'],
                          ar_reg=best_params['ar_reg'],
                          learning_rate = best_params['learning_rate'],
                          epochs = best_params['epochs'],
                          n_lags= best_params['n_lags']
                          )
model = model.add_country_holidays(country_name="US")
for column in regressed_features:
    model.add_lagged_regressor(column, n_lags=lags_for_regressed_features[column])
        
model.fit(rs, freq="D", progress="off")

new_rs = rs.copy()

new_rs = new_rs.drop(columns=['y'])

forecast = model.predict(rs)

Training: 0it [00:00, ?it/s]

Predicting: 35it [00:00, ?it/s]

## Plots and Evaluation of the Model

In [12]:
model.plot(forecast)

ERROR - (NP.plotly.plot) - plotly-resampler is not installed. Please install it to use the resampler.


In [13]:
model.plot_parameters()

ERROR - (NP.plotly.plot_parameters) - plotly-resampler is not installed. Please install it to use the resampler.
